# 06b — Variational Autoencoders

## The One-Sentence Version
A VAE doesn't just compress — it learns a smooth, structured latent space you can navigate and sample from.

## What You'll Build Intuition For
- Why regular autoencoder latent spaces can be disorganised
- How encoding to distributions (not points) creates smoothness
- The reparameterisation trick and KL divergence loss
- Interpolation and generation in latent space

## Prerequisites
- [06a — Autoencoder Basics](06a_autoencoder_basics.ipynb) — the encoder-bottleneck-decoder architecture

## The Story

A regular autoencoder learns to compress data, but its latent space can be messy. Two similar digits might end up far apart, and the space between encoded points might correspond to nothing meaningful.

A Variational Autoencoder fixes this by adding a simple constraint: instead of encoding each input to a single point, encode it to a *distribution* (a mean and variance). Then sample from that distribution before decoding. A penalty (KL divergence) pushes these distributions toward a standard normal.

The effect is dramatic: the latent space becomes smooth and continuous. You can walk between any two points and see meaningful transitions. You can sample random points and get plausible outputs. The latent space becomes a proper *map* of your data.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from utils.plotting import apply_style, COLOURS, PALETTE
from utils.data_generators import make_digit_data

apply_style()
rng = np.random.default_rng(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## The Problem with Regular Autoencoders

A regular autoencoder can place encoded points anywhere in latent space. There's no guarantee that:
- Nearby points decode to similar outputs
- The space between encoded points is meaningful
- Random samples from latent space produce valid outputs

The VAE adds structure by making the latent space probabilistic.

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 128), nn.ReLU())
        self.mu = nn.Linear(128, latent_dim)
        self.logvar = nn.Linear(128, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, input_dim), nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.mu(h), self.logvar(h)

    def reparameterise(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterise(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss(recon_x, x, mu, logvar):
    """VAE loss = reconstruction loss + KL divergence."""
    recon_loss = nn.functional.binary_cross_entropy(recon_x, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_loss

print("VAE encodes to (μ, σ) instead of a single point.")
print("The KL loss pushes these distributions toward N(0, 1).")
print("The reparameterisation trick: z = μ + σ × ε, where ε ~ N(0, 1).")

In [ ]:
# Prepare data
data, target, images = make_digit_data()
X_tensor = torch.FloatTensor(data / 16.0).to(device)
dataset = TensorDataset(X_tensor)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

# Train VAE
vae = VAE(64, 2).to(device)
optimiser = torch.optim.Adam(vae.parameters(), lr=1e-3)
vae_losses = []

for epoch in range(200):
    epoch_loss = 0
    for (batch,) in loader:
        recon, mu, logvar = vae(batch)
        loss = vae_loss(recon, batch, mu, logvar)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        epoch_loss += loss.item()
    vae_losses.append(epoch_loss / len(dataset))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(vae_losses, color=COLOURS["signal"], linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("VAE Loss (Recon + KL)")
ax.set_title("VAE training: reconstruction + regularisation")
plt.tight_layout()
plt.savefig("visuals/06b_vae_training.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Final loss: {vae_losses[-1]:.2f}")

In [ ]:
# Visualise VAE latent space
vae.eval()
with torch.no_grad():
    mu_all, _ = vae.encode(X_tensor)
    Z_vae = mu_all.cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 8))
for digit in range(10):
    mask = target == digit
    ax.scatter(Z_vae[mask, 0], Z_vae[mask, 1], s=12, alpha=0.6,
               label=str(digit), color=PALETTE[digit % len(PALETTE)])

ax.set_xlabel("z₁")
ax.set_ylabel("z₂")
ax.set_title("VAE latent space: smooth, continuous, and centred")
ax.legend(title="Digit", markerscale=3, fontsize=9)
plt.tight_layout()
plt.savefig("visuals/06b_vae_latent.png", dpi=150, bbox_inches="tight")
plt.show()

print("The KL loss pushes the latent distributions toward N(0,1),")
print("creating a smooth, continuous space centred at the origin.")

## Interpolation Demo

The smoothness of the VAE latent space means we can interpolate between two digits and see a meaningful transition.

In [ ]:
# Interpolate between a "3" and an "8"
idx_3 = np.where(target == 3)[0][0]
idx_8 = np.where(target == 8)[0][0]

vae.eval()
with torch.no_grad():
    mu_3, _ = vae.encode(X_tensor[idx_3:idx_3+1])
    mu_8, _ = vae.encode(X_tensor[idx_8:idx_8+1])

    # Linear interpolation in latent space
    n_steps = 10
    alphas = np.linspace(0, 1, n_steps)
    interpolated = []
    for alpha in alphas:
        z = (1 - alpha) * mu_3 + alpha * mu_8
        decoded = vae.decode(z).cpu().numpy()
        interpolated.append(decoded[0])

fig, axes = plt.subplots(1, n_steps, figsize=(16, 2))
for i, (ax, img) in enumerate(zip(axes, interpolated)):
    ax.imshow((img * 16).reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
    ax.set_xticks([])
    ax.set_yticks([])
    if i == 0:
        ax.set_title("3", fontsize=12, color=COLOURS["signal"])
    elif i == n_steps - 1:
        ax.set_title("8", fontsize=12, color=COLOURS["noise"])

fig.suptitle("Interpolation: smooth transition from 3 → 8 in latent space",
             fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/06b_interpolation.png", dpi=150, bbox_inches="tight")
plt.show()

print("Each intermediate image is decoded from a point between the two digits.")
print("The smooth transition shows the latent space is continuous — no gaps or jumps.")

## Generation Demo

Because the KL loss pushes the latent distributions toward N(0, 1), we can sample random points from a standard normal and decode them into plausible digit-like images.

In [ ]:
# Sample from the latent space and decode
vae.eval()
with torch.no_grad():
    z_samples = torch.randn(20, 2).to(device)
    generated = vae.decode(z_samples).cpu().numpy()

fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i in range(20):
    row, col = divmod(i, 10)
    axes[row, col].imshow((generated[i] * 16).reshape(8, 8), cmap="gray_r",
                           vmin=0, vmax=16)
    axes[row, col].set_xticks([])
    axes[row, col].set_yticks([])

fig.suptitle("Generated digits: random samples from N(0, 1) decoded into images",
             fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/06b_generation.png", dpi=150, bbox_inches="tight")
plt.show()

print("These images were never in the training data — they're generated from random noise.")
print("This only works because the KL loss made the latent space match N(0, 1).")
print("A regular autoencoder's random samples would produce garbage.")

In [ ]:
# Grid visualisation: decode a grid of points across latent space
n_grid = 15
z1 = np.linspace(-3, 3, n_grid)
z2 = np.linspace(-3, 3, n_grid)

canvas = np.zeros((n_grid * 8, n_grid * 8))

vae.eval()
with torch.no_grad():
    for i, zi in enumerate(z1):
        for j, zj in enumerate(z2):
            z = torch.FloatTensor([[zi, zj]]).to(device)
            decoded = vae.decode(z).cpu().numpy()[0]
            canvas[i*8:(i+1)*8, j*8:(j+1)*8] = (decoded * 16).reshape(8, 8)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(canvas, cmap="gray_r", vmin=0, vmax=16)
ax.set_xlabel("z₁")
ax.set_ylabel("z₂")
ax.set_xticks(np.linspace(0, n_grid * 8, 7))
ax.set_xticklabels([f"{z:.1f}" for z in np.linspace(-3, 3, 7)])
ax.set_yticks(np.linspace(0, n_grid * 8, 7))
ax.set_yticklabels([f"{z:.1f}" for z in np.linspace(-3, 3, 7)])
ax.set_title("Latent space grid: every point decodes to a digit-like image")
plt.tight_layout()
plt.savefig("visuals/06b_latent_grid.png", dpi=150, bbox_inches="tight")
plt.show()

print("The grid shows smooth transitions between digit types.")
print("This is the VAE's key advantage: a navigable, continuous latent space.")

<details>
<summary><b>The Maths Behind This</b></summary>

**VAE loss:** $\mathcal{L} = \underbrace{\mathbb{E}_{q(z|x)}[\log p(x|z)]}_{\text{reconstruction}} - \underbrace{\text{KL}(q(z|x) \| p(z))}_{\text{regularisation}}$

The encoder outputs $q(z|x) = \mathcal{N}(\mu(x), \sigma^2(x))$. The prior is $p(z) = \mathcal{N}(0, I)$.

**KL divergence** (closed form for two Gaussians):

$$\text{KL}(q \| p) = -\frac{1}{2}\sum_{j=1}^d \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

This penalises the encoder for placing latent distributions far from the origin ($\mu_j^2$ term) or making them too narrow ($\sigma_j^2$ and $\log\sigma_j^2$ terms).

**Reparameterisation trick:** To backpropagate through the sampling step, we write $z = \mu + \sigma \odot \epsilon$ where $\epsilon \sim \mathcal{N}(0, I)$. The randomness is in $\epsilon$ (which has no learnable parameters), so gradients flow through $\mu$ and $\sigma$ normally.

</details>

## Where This Matters

**Drug discovery:** VAEs learn smooth latent spaces over molecular structures. Chemists can interpolate between known drugs in latent space to find novel molecules with desired properties — the smoothness guarantees that interpolated points correspond to valid molecular structures.

**Synthetic data generation:** In healthcare, patient data is scarce and privacy-sensitive. VAEs trained on existing records can generate realistic synthetic patient data for model development, without exposing real patient information.

## Feynman Check

1. **What does the KL divergence loss do and why is it necessary?**

2. **Why can you interpolate in VAE latent space but not regular autoencoder space?**

3. **What's the reparameterisation trick and why do we need it?**

---

Now we have three compression methods: PCA (linear), autoencoders (nonlinear), and VAEs (nonlinear + structured). In **06c — Comparing to PCA**, we'll put them head-to-head and learn when each is worth the complexity.